Week 2 Notebook - Factors in Mental Health

My project looks at why some people struggle with their mental health more than others, focusing on everyday things like money, jobs, physical health and access to medical care. The goal is to see how well these factors can predict who is likely to be struggling, which could help health agencies decide where to focus support.

I'm using the CDC's Behavioral Risk Factor Surveillance System (BRFSS), a yearly phone survey of hundreds of thousands of American adults about their health and daily lives. One of its questions asks how many days in the past month a person's mental health was "not good," and that's the number I'm trying to predict. Above I did some variable work to fix some issues regarding phone responses.

In [5]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import r2_score, mean_squared_error

full_df = pd.read_sas('LLCP2024.XPT ', format='xport')
cols = ['MENTHLTH','PHYSHLTH','INCOME3','EMPLOY1','_HLTHPL2',
        'MEDCOST1','GENHLTH','_AGE_G','_BMI5']
brfss = full_df[cols].copy(); del full_df

for c in ['MENTHLTH','PHYSHLTH']:
    brfss[c] = brfss[c].replace({88: 0, 77: np.nan, 99: np.nan})
brfss['INCOME3']  = brfss['INCOME3'].replace({77: np.nan, 99: np.nan})
brfss['EMPLOY1']  = brfss['EMPLOY1'].replace({9: np.nan})
brfss['_HLTHPL2'] = brfss['_HLTHPL2'].replace({9: np.nan})
brfss['MEDCOST1'] = brfss['MEDCOST1'].replace({7: np.nan, 9: np.nan})
brfss['GENHLTH']  = brfss['GENHLTH'].replace({7: np.nan, 9: np.nan})
brfss['_BMI5']    = brfss['_BMI5'] / 100

brfss = brfss.dropna()
brfss = pd.get_dummies(brfss, columns=['EMPLOY1'], drop_first=True, dtype=int)

brfss = brfss.sample(50_000, random_state=42)
print(brfss.shape)

(50000, 15)


I built a basic prediction model and three trimmed versions of it. The trimmed versions are designed to ignore unhelpful information and avoid being fooled by random patterns. I then checked how well each one guessed people's number of bad mental health days on data it hadn't seen before.

In [3]:
X = brfss.drop(columns='MENTHLTH'); y = brfss['MENTHLTH']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size = 0.2, random_state = 42)

scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

models = {
    'OLS':        LinearRegression(),
    'Ridge':      RidgeCV(alphas = np.logspace(-3, 3, 50)),
    'Lasso':      LassoCV(cv = 5, random_state = 42),
    'ElasticNet': ElasticNetCV(l1_ratio=[.1, .5, .9], cv = 5, random_state = 42),
}

rows, coefs = [], {}
for name, m in models.items():
    m.fit(X_tr_s, y_tr)
    pred = m.predict(X_te_s)
    rows.append({'model': name,
                 'alpha': getattr(m, 'alpha_', None),
                 'R2':    r2_score(y_te, pred),
                 'RMSE':  np.sqrt(mean_squared_error(y_te, pred)),
                 'zeroed_features': int((m.coef_ == 0).sum())})
    coefs[name] = m.coef_

print(pd.DataFrame(rows).round(3))

        model    alpha     R2   RMSE  zeroed_features
0         OLS      NaN  0.201  7.570                0
1       Ridge  138.950  0.201  7.569                0
2       Lasso    0.011  0.201  7.569                2
3  ElasticNet    0.012  0.201  7.569                2


All four models scored about the same. Each explained roughly 20% of the differences between people (R² = 0.20), and their guesses were usually off by about 7.6 days (RMSE). The trimmed versions (Ridge, Lasso and Elastic Net) didn't improve on the basic model, and Lasso dropped two factors without losing accuracy, so a simpler model works just as well. Physical health, money and access to care matter, but most of what affects mental health isn't captured in this sample.